# Public-envelope sensitivity explorer

Interactively vary the model's economic and CDR parameters and watch the
**private / public / intersection / combined** return envelopes remap across
Sub-Saharan Africa, on the headline **equilibrium, net-export** basis.

Everything is a closed-form re-evaluation of five per-pixel primitives
(`AGRO`, `BAS`, `GROSS`, `LCA`, `S`) built once from the `erw-7` cost rasters, so
each slider move recomputes the whole map in milliseconds — no model re-run.
The math matches `erw/erw-12-public-envelope-sensitivity.R` exactly (validated:
central case reproduces the paper's 1.65 / 3.16 Mha public envelope).

**The levers**

| slider | symbol | what it does |
|---|---|---|
| Carbon price | $p$ | VCM credit price, \$/tCO₂ |
| MRV cost | $m$ | per-tonne measurement/verification cost, netted from price |
| Delivered-cost × | $c$ | multiplier on the full delivered basalt cost |
| CDR rate × | $r$ | multiplier on **gross** removal per tonne (reactive fraction × grain × climate) |
| Net-export λ | $\lambda$ | acidity-sink partition: 1 = sequential bound, 0 = gross accounting |
| Yield × | $y$ | multiplier on the agronomic uplift |
| Solar haulage | | fraction cut from the transport term of delivered cost |

Two structural facts the sliders make visible: the **public** envelope depends on
$p$, $m$, $c$ only through the ratio $(p-m)/c$ (move them so the ratio is constant
and the public map does not change), and durable removal is **super-linear in $r$**
(the acidity sink and lifecycle emissions are fixed subtrahends inside a `max(·,0)`).

---
**Setup.** Needs `rasterio numpy matplotlib ipywidgets` and the `data/economics_erw`
rasters. From the repo root:
```bash
python3 -m venv .venv && source .venv/bin/activate
pip install rasterio numpy matplotlib ipywidgets jupyter
ERW_ROOT=$(pwd) jupyter notebook paper/notebooks/public_envelope_explorer.ipynb
```
(In JupyterLab, `ipywidgets` works out of the box; in classic Notebook ≥ 7 too.)

In [ ]:
import os, sys, numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
import ipywidgets as W
from IPython.display import display

# locate the repo and the primitives module (paper/analysis)
ROOT = os.environ.get('ERW_ROOT', os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
sys.path.insert(0, os.path.join(ROOT, 'paper', 'analysis'))
os.environ.setdefault('ERW_ROOT', ROOT)
import erw_envelope_primitives as EP

ALLOCS = ['targeted', 'uniform_10', 'uniform_20', 'uniform_50']
print('building primitives (once per allocation) …')
PRIM = {a: EP.build(a) for a in ALLOCS}
print('ready:', ', '.join(ALLOCS))

In [ ]:
COLS = ['#7d1f8c', '#1a7d3c', '#2156a8', '#e8a33d', '#e8e8e8']   # both, priv, pub, comb-only, neither
LABS = ['both (intersection)', 'private only', 'public only', 'combined only', 'neither']
_cmap = ListedColormap(COLS); _norm = BoundaryNorm([.5,1.5,2.5,3.5,4.5,5.5], _cmap.N)

def render(alloc, carbon, mrv, cmult, rmult, lam, ymult, solar):
    P = PRIM[alloc]
    env = EP.envelopes(P, carbon=carbon, mrv=mrv, cmult=cmult, rmult=rmult, lam=lam, ymult=ymult, solar=solar)
    ar  = EP.areas(P, env)
    typ = EP.typology(P, env)
    ratio = (carbon - mrv) / cmult
    fig, ax = plt.subplots(1, 2, figsize=(13, 5.8), gridspec_kw={'width_ratios': [2, 1]})
    ax[0].imshow(typ, cmap=_cmap, norm=_norm, extent=P['extent'], interpolation='nearest')
    ax[0].set_title(f"Return typology — {alloc}, equilibrium net-export\n"
                    f"${carbon:.0f} price · ${mrv:.0f} MRV · cost×{cmult:.2f} · CDR×{rmult:.2f} · "
                    f"λ={lam:.2f} · yield×{ymult:.2f} · solar−{solar*100:.0f}%\n"
                    f"public-envelope net price / cost ratio (p−m)/c = {ratio:.0f}", fontsize=9)
    ax[0].set_xlabel('lon'); ax[0].set_ylabel('lat')
    ax[0].legend(handles=[Patch(color=c, label=l) for c, l in zip(COLS, LABS)],
                 loc='lower left', fontsize=7, framealpha=.9)
    keys = ['private', 'public', 'intersection', 'combined']
    y = np.arange(4)[::-1]
    ax[1].barh(y, [ar[k] for k in keys], color=['#1a7d3c', '#2156a8', '#7d1f8c', '#555'])
    for yi, k in zip(y, keys):
        ax[1].text(ar[k], yi, f'  {ar[k]:.2f}', va='center', fontsize=10)
    ax[1].set_yticks(y); ax[1].set_yticklabels(keys)
    ax[1].set_xlim(0, max(6, max(ar.values()) * 1.18)); ax[1].set_xlabel('area (Mha)')
    ax[1].set_title('Envelope areas', fontsize=10)
    plt.tight_layout(); plt.show()

# sanity check outside the widget
render('targeted', 150, 20, 1.0, 1.0, 1.0, 1.0, 0.0)

In [ ]:
sl = dict(continuous_update=False, style={'description_width': '130px'},
          layout=W.Layout(width='420px'))
controls = dict(
    alloc = W.Dropdown(options=ALLOCS, value='targeted', description='allocation', **{k:v for k,v in sl.items() if k!='continuous_update'}),
    carbon= W.FloatSlider(min=50, max=500, step=10, value=150, description='carbon price $/tCO2', **sl),
    mrv   = W.FloatSlider(min=0,  max=60,  step=5,  value=20,  description='MRV cost $/tCO2', **sl),
    cmult = W.FloatSlider(min=0.5,max=2.0, step=0.05,value=1.0,description='delivered cost ×', **sl),
    rmult = W.FloatSlider(min=0.25,max=3.0,step=0.05,value=1.0,description='CDR rate × (gross)', **sl),
    lam   = W.FloatSlider(min=0.0,max=1.0, step=0.05,value=1.0,description='net-export λ', **sl),
    ymult = W.FloatSlider(min=0.5,max=2.5, step=0.05,value=1.0,description='yield ×', **sl),
    solar = W.FloatSlider(min=0.0,max=0.5, step=0.05,value=0.0,description='solar haulage cut', **sl),
)
ui = W.VBox([controls['alloc'],
             W.HBox([W.VBox([controls['carbon'], controls['mrv'], controls['cmult'], controls['solar']]),
                     W.VBox([controls['rmult'], controls['lam'], controls['ymult']])])])
out = W.interactive_output(render, controls)
display(ui, out)

### Things to try

- **The public envelope has only two knobs.** Set CDR× and λ aside; raise the carbon
  price to \$300 and the delivered cost to ×2.0. The public map is *identical* to
  \$150 / ×1.0 — both give $(p-m)/c = 130$. Price and cost only ever act through that ratio.
- **Super-linearity in the CDR rate.** Push CDR× from 1.0 to 2.0: the public envelope
  roughly quadruples (targeted 1.65 → ~5.9 Mha), far more than doubling. The acidity
  sink is a fixed subtrahend, so scaling *gross* removal is convex in the net.
- **Allocation and the carbon case.** Switch to `uniform_20`: the public envelope
  roughly doubles (1.65 → 3.16 Mha) while private shrinks — the lime-requirement dose
  is CDR-minimal by construction.
- **λ barely matters.** Slide λ from 1 to 0 (gross accounting): the public envelope
  moves only ~0.6 Mha — the most-scrutinized modeling choice is the least influential lever.